# COMP 581: Interactive Linear Algebra Refresher and Differential Drive Forward Kinematics Example

This notebook is for UNC COMP 581: Introduction to Robotics. It contains a **Binder-ready** refresher on the linear algebra you’ll use in this course and robotics more generally. It then uses these concepts to in showcasing forward kinematics for differential drive robots. Some key concepts include:
- vectors and vector geometry
- dot product and projection
- matrices and matrix multiplication
- rigid-body transforms and changing coordinates between reference frames
- **homogeneous transformation matrices** in **2D** and **3D**
- differential drive robot and control parameters
- differential drive forward kinematics calculations

Throughout, use sliders and interactive widgets to build intuition by seeing the math move points, vectors, coordinate frames, and, at the end, a differential drive robot.

If you open this in Binder, remember that all your changes (if any) will be lost as soon as you close the webpage. Make sure to download your notebook if you want to save them.

---


## Environment sanity check

Run this cell first. It checks that required packages are installed and that widgets/Plotly render.


In [5]:
import importlib
import os
import sys

# If running in Google Colab, enable custom widgets and install extras as needed.
IN_COLAB = os.getenv("COLAB_RELEASE_TAG") is not None
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
    !{sys.executable} -m pip install -q ipycanvas anywidget

required = ["numpy", "matplotlib", "ipywidgets", "plotly", "ipycanvas"]
optional = ["sympy", "anywidget"]

print("Checking required packages...")
missing = [p for p in required if importlib.util.find_spec(p) is None]
for p in required:
    print(("✓ " if p not in missing else "✗ ") + p)

if missing:
    raise ImportError(f"Missing required packages: {missing}")

print("Checking optional packages...")
for p in optional:
    print(("✓ " if importlib.util.find_spec(p) is not None else "○ ") + p)

print("Widget test (you should see a slider):")
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

display(widgets.IntSlider(description="Widget test"))

print("Plotly test (you should see a 3D chart):")
# Use a regular Figure instead of FigureWidget so this also works reliably in Colab.
test_fig = go.Figure(data=[go.Scatter3d(x=[0,1], y=[0,1], z=[0,1], mode="lines+markers")])
test_fig.update_layout(title="Plotly sanity check", margin=dict(l=0,r=0,t=40,b=0))
test_fig.show()

print("Sanity check complete.")

Checking required packages...
✓ numpy
✓ matplotlib
✓ ipywidgets
✓ plotly
✓ ipycanvas
Checking optional packages...
✓ sympy
○ anywidget
Widget test (you should see a slider):


IntSlider(value=0, description='Widget test')

Plotly test (you should see a 3D chart):


Sanity check complete.


## Run the next cell to import our libraries and set up our plotting helper functions.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import plotly.graph_objects as go

def setup_2d(ax, lim=3):
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

def arrow(ax, v, label=None):
    ax.arrow(0, 0, v[0], v[1], head_width=0.12, length_includes_head=True)
    if label:
        ax.text(v[0]*1.05, v[1]*1.05, label)

def draw_frame_2d(ax, origin, theta, s=0.8, label=None):
    o = np.array(origin, dtype=float)
    c, sn = np.cos(theta), np.sin(theta)
    ex = np.array([c, sn])
    ey = np.array([-sn, c])
    ax.arrow(o[0], o[1], s*ex[0], s*ex[1], head_width=0.10, length_includes_head=True)
    ax.arrow(o[0], o[1], s*ey[0], s*ey[1], head_width=0.10, length_includes_head=True)
    if label:
        ax.text(o[0] + 0.05, o[1] + 0.05, label)

# 1) Vectors in 2D

A vector is a **direction and magnitude**. In robotics, vectors represent:
- positions (points) like $p = \begin{bmatrix} x \\ y \end{bmatrix}$
- velocities, forces, axes of motion, etc.

We will typically express positions as a column vector, which is necessary for our matrix operations as you will soon see. If a point is initially defined as a row vector, we need to take it's transpose to convert it into a column vector. So you may often see notation like this: $p = [x\ y]^T$. This simply gives us $p = \begin{bmatrix} x \\ y \end{bmatrix}$

Run the cell below and use the sliders to explore vector addition and scaling.


In [8]:
def show_vectors(scale_a=1.0, ax=0.8, ay=1.2, bx=1.0, by=0.5):
    a = np.array([ax, ay]) * scale_a
    b = np.array([bx, by])
    c = a + b

    fig, axp = plt.subplots(figsize=(6,6))
    setup_2d(axp, lim=3)
    arrow(axp, a, "a")
    arrow(axp, b, "b")
    arrow(axp, c, "a+b")
    axp.set_title("Vector addition and scaling")
    axp.text(-2.8, 2, f"a = {np.array2string(a.round(3), separator=', ')}\n b = {b.round(3)}\n a+b = {c.round(3)}",
             bbox=dict(boxstyle="round", alpha=0.15))
    plt.show()

ui = widgets.VBox([
    widgets.FloatSlider(value=1.0, min=-2, max=2, step=0.05, description="scale(a)"),
    widgets.FloatSlider(value=0.8, min=-2, max=2, step=0.05, description="a_x"),
    widgets.FloatSlider(value=1.2, min=-2, max=2, step=0.05, description="a_y"),
    widgets.FloatSlider(value=1.0, min=-2, max=2, step=0.05, description="b_x"),
    widgets.FloatSlider(value=0.5, min=-2, max=2, step=0.05, description="b_y"),
])
out = widgets.interactive_output(show_vectors, {
    "scale_a": ui.children[0],
    "ax": ui.children[1],
    "ay": ui.children[2],
    "bx": ui.children[3],
    "by": ui.children[4],
})
display(ui, out)


Output()

# 2) Dot product and angles

We didn't talk too much about the dot product in class, but it is generally useful in robotics and provides good intuition behind how matrix multiplication works. The dot product is defined as:

$$
u \cdot v = \|u\|\,\|v\|\cos\theta
$$

where $\|u\|$ indicates the magnitude of vector $u$. Remember you can calculate the magnitude of a vector by taking the square root of the sum of the squares of its components. For example, for a 2D vector $\vec{v} = (x, y)$, the magnitude is $\|v\| = \sqrt{x^2 + y^2}$

Likewise, if $u =[a\ b]$ and $v = \begin{bmatrix} c \\ d \end{bmatrix}$ then $u \cdot v = [ac + bd]$ 

Key uses in kinematics:
- computing angles between vectors/axes (using the equations above, we find that $\cos\theta = \frac {u \cdot v}{\|u\|\,\|v\|}$ )
- projection of one vector onto another (calculate the dot product and then extend it along one of the vectors, as in the example below)
- checking orthogonality (dot product = 0 when vectors are at right angles)

Run the code cell below to interactively explore how $a \cdot b$ changes with vector magnitudes and angles.


In [9]:
def dot_demo(a_len=1.5, b_len=1.5, theta_deg=45):
    theta = np.deg2rad(theta_deg)
    a = np.array([a_len, 0.0])
    b = b_len * np.array([np.cos(theta), np.sin(theta)])
    dot = float(a @ b)
    cos_th = dot / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-12)

    proj_len = dot / (np.linalg.norm(a) + 1e-12)
    proj = np.array([proj_len, 0.0])

    fig, axp = plt.subplots(figsize=(6,6))
    setup_2d(axp, lim=3)
    arrow(axp, a, "a")
    arrow(axp, b, "b")
    axp.plot([b[0], proj[0]], [b[1], proj[1]], linestyle="--", linewidth=1)
    arrow(axp, proj, "proj(b onto a)")
    axp.set_title("Dot product and projection")
    axp.text(-2.8, 2,
             f"a·b = {dot:.3f}\ncosθ = {cos_th:.3f}\nθ = {theta_deg:.1f}°",
             bbox=dict(boxstyle="round", alpha=0.15))
    plt.show()

ui = widgets.VBox([
    widgets.FloatSlider(value=1.5, min=0.2, max=2.5, step=0.05, description="|a|"),
    widgets.FloatSlider(value=1.5, min=0.2, max=2.5, step=0.05, description="|b|"),
    widgets.FloatSlider(value=45, min=-180, max=180, step=1, description="θ (deg)"),
])
out = widgets.interactive_output(dot_demo, {
    "a_len": ui.children[0],
    "b_len": ui.children[1],
    "theta_deg": ui.children[2],
})
display(ui, out)


Output()

# 3) Rotation matrices (2D)

A 2D rotation around the origin by angle $\theta$ can be accomplished by the following matrix:

$$
R(\theta) =
\begin{bmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{bmatrix}
$$

Rotations preserve lengths and angles: $\|Rv\| = \|v\|$.


In [10]:
def rot2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])

def rot_demo(theta_deg=30, vx=1.2, vy=0.6):
    theta = np.deg2rad(theta_deg)
    v = np.array([vx, vy])
    v2 = rot2(theta) @ v

    fig, axp = plt.subplots(figsize=(6,6))
    setup_2d(axp, lim=3)
    arrow(axp, v, "v")
    arrow(axp, v2, "R v")
    axp.set_title("2D rotation")
    axp.text(-2.8, 2, f"θ={theta_deg:.1f}°\n|v|={np.linalg.norm(v):.3f}\n|Rv|={np.linalg.norm(v2):.3f}",
             bbox=dict(boxstyle="round", alpha=0.15))
    plt.show()

ui = widgets.VBox([
    widgets.FloatSlider(value=30, min=-180, max=180, step=1, description="θ (deg)"),
    widgets.FloatSlider(value=1.2, min=-2, max=2, step=0.05, description="v_x"),
    widgets.FloatSlider(value=0.6, min=-2, max=2, step=0.05, description="v_y"),
])
out = widgets.interactive_output(rot_demo, {
    "theta_deg": ui.children[0], "vx": ui.children[1], "vy": ui.children[2]
})
display(ui, out)


Output()

# 4) Homogeneous transforms in 2D (rigid-body transforms)

A rigid transform combines rotation + translation. In 2D we can write this as the following homegeneous matrix:

$$
T =
\begin{bmatrix}
R & t \\
0\ 0 & 1
\end{bmatrix}
=
\begin{bmatrix}
\cos\theta & -\sin\theta & t_x \\
\sin\theta & \cos\theta & t_y \\
0 & 0 & 1
\end{bmatrix}
$$

To transform a point $p = [x\ y]^T$ with a homogeneous transform, use homogeneous coordinates:

$$
\tilde{p} = \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},\quad
\tilde{p}' = T\,\tilde{p}.
$$

If $T = {}^{A}T_{B}$, (i.e., $T$ is a matrix that expresses the transformation from reference frame ${B}$ to reference frame ${A}$), then multiplying $p_B$ (a point defined relative to reference frame ${B}$) gives the **same physical point** expressed from the point of view of reference of frame ${A}$:
$$
p_A = {}^{A}T_{B}\,p_B.
$$

The inverse transform converts coordinates the other way:
$$
p_B = ({}^{A}T_{B})^{-1} p_A = {}^{B}T_{A}\,p_A.
$$

Run the code below to show an interactive visualization of how a homogeneous transform can be used to convert a point from one reference frame to another.


In [11]:
def T2(theta, tx, ty):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, tx],
                     [s,  c, ty],
                     [0,  0,  1]], dtype=float)

def apply_T2(Tm, p):
    ph = np.array([p[0], p[1], 1.0])
    q = Tm @ ph
    return q[:2]

def demo_T2(theta_deg=30, tx=0.5, ty=0.2, px=1.0, py=0.5, show_frames=True):
    theta = np.deg2rad(theta_deg)
    T_AB = T2(theta, tx, ty)

    p_B = np.array([px, py])
    p_A = apply_T2(T_AB, p_B)

    fig, axp = plt.subplots(figsize=(6,6))
    setup_2d(axp, lim=3)

    if show_frames:
        draw_frame_2d(axp, origin=(0,0), theta=0.0, label="{A}")
        draw_frame_2d(axp, origin=(tx,ty), theta=theta, label="{B}")

    axp.scatter([p_A[0]], [p_A[1]], s=80)
    axp.text(p_A[0]+0.05, p_A[1]+0.05, "p (in A)")

    axp.set_title("2D rigid transform:  $p_A = ^A T_B p_B$")
    axp.text(-2.8, 1,
             f"θ={theta_deg:.1f}°, t=({tx:.2f},{ty:.2f})\n"
             f"$p_B$=({p_B[0]:.2f},{p_B[1]:.2f})\n"
             f"$p_A$=({p_A[0]:.2f},{p_A[1]:.2f})\n"
             f"$^A T_B$=\n{np.round(T_AB, decimals=2)}",
             bbox=dict(boxstyle="round", alpha=0.15))
    plt.show()

ui = widgets.VBox([
    widgets.FloatSlider(value=30, min=-180, max=180, step=1, description="θ (deg)"),
    widgets.FloatSlider(value=0.5, min=-2, max=2, step=0.05, description="t_x"),
    widgets.FloatSlider(value=0.2, min=-2, max=2, step=0.05, description="t_y"),
    widgets.FloatSlider(value=1.0, min=-2, max=2, step=0.05, description="p_x in B"),
    widgets.FloatSlider(value=0.5, min=-2, max=2, step=0.05, description="p_y in B"),
    widgets.Checkbox(value=True, description="Show reference frame axes"),
])
out = widgets.interactive_output(demo_T2, {
    "theta_deg": ui.children[0], "tx": ui.children[1], "ty": ui.children[2],
    "px": ui.children[3], "py": ui.children[4], "show_frames": ui.children[5]
})
display(ui, out)


Output()

# 5) Homogeneous transforms in 3D

In 3D, rigid transforms use 4×4 matrices:

$$
{}^{A}T_{B} =
\begin{bmatrix}
{}^{A}R_{B} & {}^{A}t_{B} \\
0\ 0\ 0 & 1
\end{bmatrix}
$$

We’ll visualize a frame ${B}$ rotated and translated relative to ${A}$, and a point expressed in ${B}$.  
We can then use a 4×4 homogeneous transformation matrix $^{A}T_{B}$ to find the point's coordinates in ${A}$.

In 2D, there is only one option for the rotation component of the homogeneous transformation matrix. In 3D, there are multiple options, depending on which axis we rotate around (x, y, or z) and which order we perform the rotations. In the example below, we use a yaw–pitch–roll rotation:
$$
R = R_z(\text{yaw})\,R_y(\text{pitch})\,R_x(\text{roll}).
$$

Run the code cell below to see an interactive 3D rotation example:

In [12]:
def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

def T3(R, t):
    Tm = np.eye(4)
    Tm[:3,:3] = R
    Tm[:3,3] = t
    return Tm

def apply_T3(Tm, p):
    ph = np.array([p[0], p[1], p[2], 1.0])
    q = Tm @ ph
    return q[:3]

def frame_traces_3d(origin, R, axis_len=0.5):
    o = origin
    ex, ey, ez = R[:,0], R[:,1], R[:,2]
    xt, yt, zt = o + axis_len*ex, o + axis_len*ey, o + axis_len*ez
    traces = []
    traces.append(go.Scatter3d(x=[o[0], xt[0]], y=[o[1], xt[1]], z=[o[2], xt[2]], mode="lines", name="x-axis"))
    traces.append(go.Scatter3d(x=[o[0], yt[0]], y=[o[1], yt[1]], z=[o[2], yt[2]], mode="lines", name="y-axis"))
    traces.append(go.Scatter3d(x=[o[0], zt[0]], y=[o[1], zt[1]], z=[o[2], zt[2]], mode="lines", name="z-axis"))
    traces.append(go.Scatter3d(x=[xt[0]], y=[xt[1]], z=[xt[2]], mode="text", text=["x"], showlegend=False))
    traces.append(go.Scatter3d(x=[yt[0]], y=[yt[1]], z=[yt[2]], mode="text", text=["y"], showlegend=False))
    traces.append(go.Scatter3d(x=[zt[0]], y=[zt[1]], z=[zt[2]], mode="text", text=["z"], showlegend=False))
    return traces

def make_3d_transform_figure(roll_deg, pitch_deg, yaw_deg, txv, tyv, tzv, pxv, pyv, pzv):
    r = np.deg2rad(roll_deg)
    p = np.deg2rad(pitch_deg)
    y = np.deg2rad(yaw_deg)
    R = Rz(y) @ Ry(p) @ Rx(r)
    t = np.array([txv, tyv, tzv])
    T_AB = T3(R, t)

    p_B = np.array([pxv, pyv, pzv])
    p_A = apply_T3(T_AB, p_B)

    fig = go.Figure()
    fig.update_layout(
        title="3D rigid transform: p<sub>A</sub> = <sup>A</sup>T<sub>B</sub> p<sub>B</sub>",
        scene=dict(
            xaxis=dict(range=[-3,3], title="x"),
            yaxis=dict(range=[-3,3], title="y"),
            zaxis=dict(range=[-3,3], title="z"),
            aspectmode="cube"
        ),
        margin=dict(l=0,r=0,t=40,b=0),
        showlegend=False,
    )

    # Fixed world frame A
    for tr in frame_traces_3d(np.zeros(3), np.eye(3), axis_len=0.8):
        fig.add_trace(tr)

    # Moving frame B
    for tr in frame_traces_3d(t, R, axis_len=0.8):
        fig.add_trace(tr)

    # Point expressed in A after applying the transform
    fig.add_trace(
        go.Scatter3d(
            x=[p_A[0]], y=[p_A[1]], z=[p_A[2]],
            mode="markers+text", text=["p"], textposition="top center",
            showlegend=False
        )
    )

    return fig, p_B, p_A, T_AB

roll = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description="roll (deg)")
pitch = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description="pitch (deg)")
yaw = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description="yaw (deg)")
tx = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.05, description="t_x")
ty = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description="t_y")
tz = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, description="t_z")
px = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.05, description="p_x in B")
py = widgets.FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.05, description="p_y in B")
pz = widgets.FloatSlider(value=0.3, min=-2.0, max=2.0, step=0.05, description="p_z in B")

plot_out = widgets.Output()
text_out = widgets.Output()

controls = [roll, pitch, yaw, tx, ty, tz, px, py, pz]

def redraw_3d(*_):
    fig, p_B, p_A, T_AB = make_3d_transform_figure(
        roll.value, pitch.value, yaw.value,
        tx.value, ty.value, tz.value,
        px.value, py.value, pz.value
    )

    with plot_out:
        plot_out.clear_output(wait=True)
        display(fig)

    with text_out:
        text_out.clear_output(wait=True)
        print("p_B =", np.round(p_B, 2))
        print("p_A =", np.round(p_A, 2))
        print("Transformation matrix (rounded) =")
        print(np.round(T_AB, 2))

for w in controls:
    w.observe(redraw_3d, names="value")

ui = widgets.VBox([
    widgets.HBox([roll, pitch, yaw]),
    widgets.HBox([tx, ty, tz]),
    widgets.HBox([px, py, pz])
])

display(ui, plot_out, text_out)
redraw_3d()

Output()

Output()

# 6) Quick practice questions

Before moving on, test your knowledge with the following questions:

1. In 2D, imagine a point $p_B=(1,0)$ in a reference frame ${B}$, where ${B}$ is translated $t=(1,0)$ and rotated $\theta=90^\circ$ from reference frame ${A}$ and ${A}$ is our standard Cartesian coordinate frame. What is $p_A$? Verify with the widget (see #4 above). What does this tell you about how the homogeneous transformation encodes translations and rotations (i.e., does it encode a rotation followed by a translation or the other way around?)
2. In 3D (example #5 above), keep translation $t=0$ and rotate yaw from $0$ to $90^\circ$. Which axis does the point rotate around in world coordinates?
3. Compute $p_B = ({}^{A}T_{B})^{-1} p_A$ for a point you choose. Verify this in 2D with example #4 above. (Hint: for rigid transforms, $R^{-1} = R^T$).



# 7) Differential Drive Forward Kinematics

Now let's put some of these linear algebra fundamentals into practice by learning how to calculate **differential drive** forward kinematics with interactive demos.

We will:
- derive body-frame velocities from wheel speeds
- integrate robot pose $(x, y, \theta)$ forward in time
- visualize trajectories and special cases (straight, in-place rotation, arcs)

A differential drive robot has two wheels separated by track width (axle length) $L$ and wheel radius $r$:

![Alt text](diff-drive.png)

We might have two differently sized wheels, in which case we could designate $r_L$ as the left wheel radius and $r_R$ as the right wheel radius. In the interactive example below, we assume they are the same size, denoted $r$, thus $r_L = r_R = r$

Let wheel **angular** velocities (how fast each wheel is spinning) be:
- $u_R$ (right wheel), $u_L$ (left wheel) in rad/s.

You could determine these values on your real robot by querying the motor encoders and determining how much each motor had rotated over some time interval you measure via a timer or stopwatch.

Wheel **linear** speeds can be calculated from the wheel radius and angular velocities:
- $v_R = r_R\,u_R,\qquad v_L = r_L\,u_L$

Body-frame forward speed (at the measuring point located at the midpoint between wheels) and angular velocity (yaw rotation rate):
- $v = \frac{v_R + v_L}{2},\qquad \omega = \frac{v_R - v_L}{L}$

The wheel velocities determine which of three potential types of motion the robot exhibits:
- $v_R = v_L$ → straight line motion ($\omega=0$)
- $v_R = -v_L$ → in-place rotation ($v=0$)
- one wheel faster → arc motion

Run the code cell below and move the sliders and observe how $v$ and $\omega$ change. As noted above, in this demo, we assume both wheels have the same radius $r$.

In [13]:
def vw_from_wheels(omega_L, omega_R, r, L):
    vL = r * omega_L
    vR = r * omega_R
    v = 0.5 * (vR + vL)
    w = (vR - vL) / L
    return v, w, vL, vR

omegaL = widgets.FloatSlider(value=2.0, min=-10, max=10, step=0.1, description="u_L (rad/s)")
omegaR = widgets.FloatSlider(value=4.0, min=-10, max=10, step=0.1, description="u_R (rad/s)")
r = widgets.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.005, description="r (m)")
L = widgets.FloatSlider(value=0.30, min=0.10, max=0.80, step=0.01, description="L (m)")
out = widgets.Output()

def update_vw(omega_L, omega_R, r, L):
    v, w, vL, vR = vw_from_wheels(omega_L, omega_R, r, L)
    with out:
        out.clear_output()
        print(f"v_L = {vL:.3f} m/s   v_R = {vR:.3f} m/s")
        print(f"v   = {v:.3f} m/s")
        print(f"ω   = {w:.3f} rad/s  ({np.rad2deg(w):.2f} deg/s)")
        if abs(w) < 1e-8:
            print("Motion type: straight line (ω≈0)")
        elif abs(v) < 1e-8:
            print("Motion type: in-place rotation (v≈0)")
        else:
            print(f"Motion type: circular arc")

ui = widgets.VBox([widgets.HBox([omegaL, omegaR]), widgets.HBox([r, L])])
widgets.interactive_output(update_vw, {"omega_L": omegaL, "omega_R": omegaR, "r": r, "L": L})
display(ui, out)
update_vw(omegaL.value, omegaR.value, r.value, L.value)


Output()

## Calculating our new pose

Our robot pose is $\mathbf{x} = (x, y, \theta)$ in the world frame. We want to calculate our new pose $\mathbf{x} = (\dot{x}, \dot{y}, \dot{\theta})$ assuming a constant $(v,\omega)$ over some (small) time interval $\delta t$.

As we saw above, we have three potential types of motion:

1. If $\omega = 0$, it reduces to straight-line motion. In this case:
- $\dot{x} = x + v\cos\theta \delta t$
- $\dot{y} = y + v\sin\theta \delta t$
- $\dot{\theta} = \theta$ 

2. If $v_R = -v_L$, the robot is rotating in place. In this case:
- $\dot{x} = x $
- $\dot{y} = y $
- $\dot{\theta} = \theta + \alpha$ where $\alpha = \omega \delta t$

3. If neither of the two cases above apply, we are in the general case of modeling the robot's motion as an arc around a circle, where the center of the circle is the ICC and the radius is $R$.
- In class we derived that $R = \frac{L}{2} \frac{v_L + v_R}{v_R - v_L}$
- We then derived that the coordinates of the ICC can be located at $ICC = (x - R\sin\theta, y + R\cos\theta)$
- From here, you need to dervive an equation for updating your robot's pose $\dot{x}$ and $\dot{y}$. $\dot{\theta}$ is updated as before.
- $\dot{x} = ? $
- $\dot{y} = ? $
- $\dot{\theta} = \theta + \alpha$ where $\alpha = \omega \delta t$
- Hint: to determine a formula for calculating our new $(\dot{x},\dot{y})$, try multiplying out the appropriate matrices and see what you end up with. Remember our approach follows the linear algebra for rotating a point (in this case, our robot) by some amount ($\alpha$) around some other point (in this case our ICC), where we transform the system so that the ICC becomes the origin, use a rotation matrix as normal, and then transform the system back: $$T(ICC_x, ICC_y)R(\alpha)T(-ICC_x,-ICC_y)(x,y,1)^T$$
In the formula above, $T$ means a purely translational homogeneous transformation matrx and $R$ is a rotational homogeneous transformation matrix.

Run the code cell below to interactively explore and visualize these formulas in practice and better understand differential-drive forward kinematics. Try changing the initial pose, wheel speeds, and robot parameters. You'll see the default values are the same as the practice problem we worked through and solved by hand in class. Try out various types of all three motion categories (straight, spin in place, arc motions) and work to build your intuitions about what motions you'll see based on various parameter settings.


In [14]:
from ipywidgets import VBox, FloatText, Layout, interactive_output, Label, HBox, FloatSlider

def wrap_to_pi(angle):
    # Wrap angle to (-pi, pi]
    return (angle + np.pi) % (2*np.pi) - np.pi

def diffdrive_forward_kinematics(x, y, theta, u_l, u_r, dt, r, L):
    # v, omega for a differential-drive robot
    v = (r/2.0) * (u_l + u_r)
    omega = (r/L) * (u_r - u_l)

    eps = 1e-9
    if abs(omega) < eps:
        # Straight-line motion
        x1 = x + v * dt * np.cos(theta)
        y1 = y + v * dt * np.sin(theta)
        theta1 = wrap_to_pi(theta)
        return {"v": v, "omega": omega, "R": np.inf, "icc": None, "final_pose": (x1, y1, theta1)}

    R = v / omega
    icc_x = x - R * np.sin(theta)
    icc_y = y + R * np.cos(theta)

    # Closed-form integration (rotation about ICC)
    dtheta = omega * dt
    cosd, sind = np.cos(dtheta), np.sin(dtheta)

    x_rel = x - icc_x
    y_rel = y - icc_y

    x1 = cosd * x_rel - sind * y_rel + icc_x
    y1 = sind * x_rel + cosd * y_rel + icc_y
    theta1 = wrap_to_pi(theta + dtheta)

    return {"v": v, "omega": omega, "R": R, "icc": (icc_x, icc_y), "final_pose": (x1, y1, theta1)}


def simulate_path(x, y, theta, u_l, u_r, dt, r, L, steps=200):
    # Sample x(t), y(t), theta(t) over [0, dt]
    t = np.linspace(0, dt, steps)
    v = (r/2.0) * (u_l + u_r)
    omega = (r/L) * (u_r - u_l)

    eps = 1e-9
    if abs(omega) < eps:
        xs = x + v * t * np.cos(theta)
        ys = y + v * t * np.sin(theta)
        thetas = np.full_like(t, wrap_to_pi(theta))
        return t, xs, ys, thetas

    R = v / omega
    icc_x = x - R * np.sin(theta)
    icc_y = y + R * np.cos(theta)

    thetas = wrap_to_pi(theta + omega * t)
    xs = icc_x + R * np.sin(thetas)
    ys = icc_y - R * np.cos(thetas)
    return t, xs, ys, thetas


def draw_robot(ax, x, y, theta, L=2.0, body_radius=0.35, label=None):
    # Simple robot: circle + heading arrow
    body = plt.Circle((x, y), body_radius, fill=False, linewidth=2)
    ax.add_patch(body)

    arrow_len = max(0.75, 0.35 * L)
    dx = arrow_len * np.cos(theta)
    dy = arrow_len * np.sin(theta)
    ax.arrow(
        x, y, dx, dy,
        head_width=0.18*arrow_len,
        head_length=0.22*arrow_len,
        length_includes_head=True
    )

    if label is not None:
        ax.text(x, y + 1.15*body_radius, label, ha="center", va="bottom")


def format_numbers(v, omega, R, icc, x1, y1, theta1):
    if np.isinf(R):
        R_str = "∞ (straight line)"
    else:
        R_str = f"{R:.2f} m"
    icc_str = "None (straight line)" if icc is None else f"({icc[0]:.2f}, {icc[1]:.2f})"
    return f"""
**Computed values**
- v = {v:.2f} m/s
- ω = {omega:.2f} rad/s
- R = {R_str}
- ICC = {icc_str}
- new pose [x',y',theta'] = [{x1:.2f},{y1:.2f},{theta1:.2f}]
"""


def update_plot(x, y, theta, u_l, u_r, dt, r, L, steps):
    out = diffdrive_forward_kinematics(x, y, theta, u_l, u_r, dt, r, L)
    t, xs, ys, thetas = simulate_path(x, y, theta, u_l, u_r, dt, r, L, steps=int(steps))

    v, omega, R, icc = out["v"], out["omega"], out["R"], out["icc"]
    x1, y1, theta1 = out["final_pose"]

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot(xs, ys, linewidth=2)

    draw_robot(ax, x, y, theta, L=L, label="start")
    draw_robot(ax, x1, y1, theta1, L=L, label="end")

    if icc is not None and not np.isinf(R):
        ax.scatter([icc[0]], [icc[1]])
        ax.plot([icc[0], x], [icc[1], y], linestyle="--", linewidth=1.5)
        ax.text(icc[0], icc[1], "  ICC", ha="left", va="center")

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title("Differential Drive Motion Over dt")

    allx = np.r_[xs, [x, x1]]
    ally = np.r_[ys, [y, y1]]
    pad = 1.5
    ax.set_xlim(allx.min() - pad, allx.max() + pad)
    ax.set_ylim(ally.min() - pad, ally.max() + pad)
    ax.grid(True, alpha=0.3)

    plt.show()
    display(Markdown(format_numbers(v, omega, R, icc, x1, y1, theta1)))

# Defaults (matches class problem)
default_x = 2.0
default_y = 6.0
default_theta = np.pi
default_u_l = -np.pi
default_u_r = -(3/2)*np.pi
default_dt = 4.0
default_r = 0.5
default_L = 2.0

controls_left = VBox([
    Label("Initial pose"),
    FloatText(value=default_x, description="x (m):", layout=Layout(width="260px")),
    FloatText(value=default_y, description="y (m):", layout=Layout(width="260px")),
    FloatText(value=round(default_theta,3), description="θ (rad):", layout=Layout(width="260px")),
])

controls_mid = VBox([
    Label("Wheel angular velocities"),
    FloatText(value=round(default_u_l,3), description="u_l (rad/s):", layout=Layout(width="260px")),
    FloatText(value=round(default_u_r,3), description="u_r (rad/s):", layout=Layout(width="260px")),
    FloatText(value=default_dt, description="dt (s):", layout=Layout(width="260px")),
])

controls_right = VBox([
    Label("Robot parameters"),
    FloatText(value=default_r, description="r (m):", layout=Layout(width="260px")),
    FloatText(value=default_L, description="L (m):", layout=Layout(width="260px")),
    FloatSlider(value=220, min=50, max=600, step=10, description="steps:", layout=Layout(width="260px")),
])

ui = HBox([controls_left, controls_mid, controls_right], layout=Layout(justify_content="space-between"))

w = {
    "x": controls_left.children[1],
    "y": controls_left.children[2],
    "theta": controls_left.children[3],
    "u_l": controls_mid.children[1],
    "u_r": controls_mid.children[2],
    "dt": controls_mid.children[3],
    "r": controls_right.children[1],
    "L": controls_right.children[2],
    "steps": controls_right.children[3],
}

out = interactive_output(update_plot, w)
display(ui, out)

Output()

## Self-check

- Does $R$ change when rotating in place clockwise vs counter clockwise? Why?
- Try increasing **L** and leave all other parameters the same. You will see that the wider wheelbase reduces angular velocity for the same wheel speed difference. Why is this the case?
- What are various ways you can make the robot arc clockwise vs counter clockwise?
- Try setting a few parameters and calculating the final pose by hand and drawing it yourself, before checking your answer with the tool above.